# **Setup**

In [19]:
import os
import random
import shutil
from pathlib import Path
import yaml
import cv2
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

In [ ]:
BASE_DATASET = Path("/kaggle/input/ua-detrac-dataset/content/UA-DETRAC/DETRAC_Upload")

# Fallback paths (in case structure differs)
if not BASE_DATASET.exists():
    BASE_DATASET = Path("/kaggle/input/ua-detrac-dataset/DETRAC_Upload")
if not BASE_DATASET.exists():
    BASE_DATASET = Path("/kaggle/input/ua-detrac-dataset")

print(f"Using dataset root: {BASE_DATASET}")
assert BASE_DATASET.exists(), "Dataset not found! Check Kaggle input."

train_img_dir = BASE_DATASET / "images/train"
val_img_dir   = BASE_DATASET / "images/val"
train_lbl_dir = BASE_DATASET / "labels/train"
val_lbl_dir   = BASE_DATASET / "labels/val"

print(f"Train images: {len(list(train_img_dir.glob('*.jpg')))}")
print(f"Val images:   {len(list(val_img_dir.glob('*.jpg')))}")

In [ ]:
from tqdm.notebook import tqdm
def sample_half_dataset(src_img_dir: Path, src_lbl_dir: Path, dst_img_dir: Path, dst_lbl_dir: Path, fraction=0.5, seed=42):
    random.seed(seed)
    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)
    
    images = list(src_img_dir.glob("*.jpg"))
    random.shuffle(images)
    selected = images[:int(len(images) * fraction)]
    
    print(f"Copying {len(selected)} images from {src_img_dir.name} → {dst_img_dir.name}...")
    
    copied = 0
    for img_path in tqdm(selected, desc=f"Copying {src_img_dir.parent.name}", leave=False):
        lbl_path = src_lbl_dir / (img_path.stem + ".txt")
        if lbl_path.exists():
            shutil.copy(img_path, dst_img_dir / img_path.name)
            shutil.copy(lbl_path, dst_lbl_dir / lbl_path.name)
            copied += 1
    
    print(f"Done! Copied {copied}/{len(selected)} images+labels")

# Create working directory
WORK_DIR = Path("/kaggle/working/detrac_50percent")
(WORK_DIR / "images/train").mkdir(parents=True, exist_ok=True)
(WORK_DIR / "images/val").mkdir(parents=True, exist_ok=True)
(WORK_DIR / "labels/train").mkdir(parents=True, exist_ok=True)
(WORK_DIR / "labels/val").mkdir(parents=True, exist_ok=True)

# Run with progress bars
sample_half_dataset(train_img_dir, train_lbl_dir,
                    WORK_DIR / "images/train", WORK_DIR / "labels/train", fraction=0.5)

sample_half_dataset(val_img_dir, val_lbl_dir,
                    WORK_DIR / "images/val", WORK_DIR / "labels/val", fraction=0.5)

# **Preprocessing**

In [ ]:
def fix_detrac_labels(label_dir: Path):
    label_dir = Path(label_dir)
    txt_files = list(label_dir.glob("*.txt"))
    print(f"Fixing {len(txt_files)} label files in {label_dir}...")
    
    fixed = 0
    for txt_file in tqdm(txt_files, desc="Fixing labels", leave=False):
        content = txt_file.read_text().strip()
        if '\n' in content or len(content.split()) % 5 != 0:
            continue  # already fixed or malformed
        
        values = content.split()
        if len(values) % 5 != 0:
            continue
            
        fixed_lines = '\n'.join(' '.join(values[i:i+5]) for i in range(0, len(values), 5))
        txt_file.write_text(fixed_lines + '\n')
        fixed += 1
    
    print(f"Fixed {fixed}/{len(txt_files)} label files → now one object per line!")

fix_detrac_labels(WORK_DIR / "labels/train")
fix_detrac_labels(WORK_DIR / "labels/val")

In [ ]:
yaml_data = {
    "path": str(WORK_DIR),
    "train": "images/train",
    "val": "images/val",
    "nc": 4,
    "names": ["others", "car", "van", "bus"]
}

yaml_path = WORK_DIR / "detrac_50percent.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(yaml_data, f, default_flow_style=False)

print(f"YAML created → {yaml_path}")

In [ ]:
print(f"\nFinal subset → Train: {len(list((WORK_DIR/'images/train').glob('*.jpg')))} | Val: {len(list((WORK_DIR/'images/val').glob('*.jpg')))}")

# Show one fixed label example
sample_lbl = next((WORK_DIR / "labels/train").glob("*.txt"))
print(f"\nFixed label example ({sample_lbl.name}):")
!head -5 {sample_lbl}

# Visualize one image with boxes
img_file = random.choice(list((WORK_DIR / "images/train").glob("*.jpg")))
img = cv2.cvtColor(cv2.imread(str(img_file)), cv2.COLOR_BGR2RGB)
h, w = img.shape[:2]
lbl_file = WORK_DIR / "labels/train" / (img_file.stem + ".txt")

if lbl_file.exists():
    with open(lbl_file) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                cx = float(parts[1]) * w
                cy = float(parts[2]) * h
                bw = float(parts[3]) * w
                bh = float(parts[4]) * h
                x1, y1 = int(cx - bw/2), int(cy - bh/2)
                x2, y2 = int(cx + bw/2), int(cy + bh/2)
                cv2.rectangle(img, (x1,y1), (x2,y2), (255,0,0), 2)

plt.figure(figsize=(12,8))
plt.imshow(img)
plt.title(f"Fixed labels working! → {img_file.name}")
plt.axis("off")
plt.show()

# **Training the YOLOv8**

In [ ]:
#Install YOLOv8 and Pin NumPy "numpy<2"

# This prevents the NumPy 2.x incompatibility with matplotlib.
!pip install ultralytics "numpy<2"

**Create Data YAML and Train YOLOv8n**

In [ ]:
import torch.nn as nn
from ultralytics.nn import tasks

# MLCA & SEAM definitions
class MLCA(nn.Module):
    def __init__(self, c1, r=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(c1, c1//r, 1, bias=False), nn.ReLU(inplace=True),
            nn.Conv2d(c1//r, c1, 1, bias=False), nn.Sigmoid()
        )
        self.local_mix = nn.Sequential(nn.Conv2d(c1,c1,1,bias=False), nn.BatchNorm2d(c1), nn.Sigmoid())
    def forward(self, x): return x * self.fc(self.avg_pool(x)) * self.local_mix(x)

class SEAM(nn.Module):
    def __init__(self, c1, reduction=16):
        super().__init__()
        self.conv1 = nn.Conv2d(c1, c1, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(c1)
        self.dw_conv = nn.Conv2d(c1, c1, 3, 1, 1, groups=c1, bias=False)
        self.point_conv = nn.Conv2d(c1, c1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(c1)
        self.act = nn.SiLU()
        self.sigmoid = nn.Sigmoid()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(c1, c1//reduction, bias=False), nn.ReLU(inplace=True),
            nn.Linear(c1//reduction, c1, bias=False), nn.Sigmoid()
        )
    def forward(self, x):
        res = x
        out = self.sigmoid(self.bn2(self.point_conv(self.dw_conv(self.act(self.bn1(self.conv1(x)))))))
        out = x * out
        b, c, _, _ = out.size()
        return res + out * self.fc(self.avg_pool(out).view(b,c)).view(b,c,1,1)

tasks.MLCA = MLCA
tasks.SEAM = SEAM

In [23]:
# Save the improved model YAML
improved_yaml = """
nc: 4
scales:
  n: [0.33, 0.25, 1024]
backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 3, C2f, [128, True]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 6, C2f, [256, True]]
  - [-1, 1, MLCA, [64]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 6, C2f, [512, True]]
  - [-1, 1, MLCA, [128]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 3, C2f, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]
  - [-1, 1, MLCA, [256]]
head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 8], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 5], 1, Concat, [1]]
  - [-1, 3, C2f, [256]]
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 2], 1, Concat, [1]]
  - [-1, 3, C2f, [128]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [[-1, 18], 1, Concat, [1]]
  - [-1, 3, C2f, [256]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 15], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]
  - [-1, 3, C2f, [1024]]
  - [21, 1, SEAM, [32]]
  - [24, 1, SEAM, [64]]
  - [27, 1, SEAM, [128]]
  - [30, 1, SEAM, [256]]
  - [[31, 32, 33, 34], 1, Detect, [nc]]
"""

with open("/kaggle/working/yolov8_improved_p2.yaml", "w") as f:
    f.write(improved_yaml)

### **Run Training**

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")                          
model = YOLO("/kaggle/working/yolov8_improved_p2.yaml")  
model.load("yolov8n.pt")                         

results = model.train(
    data=str(yaml_path),
    epochs=100,
    imgsz=800,
    batch=16,
    patience=10,
    scale=0.9,
    mixup=0.5,
    mosaic=1.0,
    degrees=15,
    translate=0.2,
    freeze=10,
    project='/kaggle/working/runs"',
    name='yolov8_improved_p2',

    plots=True,
    exist_ok=True, 
    amp=True,
    cache=True
)

#  **Evaluate the Model**

In [ ]:
from ultralytics import YOLO

# trained weights
model = YOLO('detrac_p2_project/yolov8_improved_p2/weights/best.pt')

#Validate
metrics = model.val(
    data='/content/Detrac_File/detrac.yaml',
    imgsz=640,
    batch=8
)